In [1]:
# Install required packages in Google Colab
# !pip install -qq weaviate-client python-dotenv weaviate-agents datasets

# Weaviate Query Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saskinosie/weaviate_query_agent_demo/blob/main/weaviate_query_agent.ipynb)

Here, we'll walk you through a comprehensive example showcasing the **Weaviate Query Agent** functionality.  
The Query Agent is an intelligent layer that sits on top of your Weaviate vector database, using generative AI to optimize natural language queries and automatically determine the best search strategy.

### How the Query Agent Works

The Query Agent intelligently handles complex queries by:
- **Query Optimization**: Uses generative AI to transform natural language into optimized vector database queries
- **Collection Selection**: Automatically determines which collection(s) to search based on your question
- **Smart Filtering**: Decides when filtering is needed to narrow down results
- **Aggregation Logic**: Determines if aggregation operations should be performed (counts, grouping, etc.)
- **Execution**: Runs the optimized queries against your Weaviate instance

### Dataset Overview

This notebook demonstrates the Query Agent using three diverse collections:
- **Books**: 10,000 books with titles, authors, descriptions, and genres
- **Brands**: 104 clothing brands with descriptions, ratings, and company information  
- **Ecommerce**: 448 fashion items with prices, categories, reviews, and brand associations

All datasets come pre-vectorized with **Snowflake Arctic Embed v2.0** embeddings via Weaviate's embedding service.

### Further Reading

- Check out the datasets on [HuggingFace](https://huggingface.co/datasets/weaviate/agents)
- Learn more about Query Agents in the [official documentation](https://docs.weaviate.io/agents/query)
- Explore the technical implementation in our [Query Agent tutorial](https://docs.weaviate.io/agents/query/tutorial-ecommerce)
- Understand vector databases in the [Weaviate developer docs](https://weaviate.io/developers/weaviate)

If running locally, to ensure smooth execution and prevent potential conflicts with your global Python environment, we recommend following the instructions in the README running the code in a virtual environment.

## Libraries/packages used

The following libraries are used in this notebook:

* [<code style="color:blue;">weaviate-client[agents]:</code>](https://weaviate.io/developers/weaviate/client-libraries/python) A powerful vector database with Query Agent functionality for intelligent search
* [<code style="color:blue;">datasets:</code>](https://huggingface.co/docs/datasets/) Hugging Face datasets library for loading pre-vectorized data
* [<code style="color:blue;">os:</code>](https://docs.python.org/3/library/os.html) Used for environment variable management
* [<code style="color:blue;">dotenv:</code>](https://pypi.org/project/python-dotenv/) Loads environment variables from .env files

The packages mentioned above are already installed via the requirements.txt in the README. If you are running this in Colab or would like to manually install these packages, uncomment and run the first cell in this notebook:

* `pip install -qq weaviate-client python-dotenv weaviate-agents datasets`

<a id='TOC'></a>  
## Table of contents  

1. <a href="#Dependencies">Dependencies</a><br>
2. <a href="#Connecting">Connecting to your Weaviate cluster</a><br>
3. <a href="#Collections">Setting up collections and loading data</a><br>
   3.1 <a href="#BrandsCollection">Create and populate Brands collection</a><br>
   3.2 <a href="#EcommerceCollection">Create and populate Ecommerce collection</a><br>
   3.3 <a href="#BooksCollection">Create and populate Books collection</a><br>
4. <a href="#QueryAgent">Creating the Query Agent</a><br>
5. <a href="#ExampleQueries">Example Query Agent interactions</a><br>
   5.1 <a href="#ClassicRAG">Classic RAG search</a><br>
   5.2 <a href="#DatabaseFiltering">Generated database filtering</a><br>
   5.3 <a href="#StatisticalQueries">Statistical and aggregation queries</a><br>
   5.4 <a href="#MultipleQueries">Querying multiple databases</a><br>
   5.5 <a href="#MissingInfo">Identifying missing information</a><br>
6. <a href="#ContractAnalysis">Practical Business Use Case for the Query Agent: Contract Analysis</a><br>
   6.1 <a href="#AgentTask1">Agent Task 1: Risk Assessment</a><br>
   6.2 <a href="#AgentTask2">Agent Task 2: Contract Comparison</a><br>
   6.3 <a href="#AgentTask3">Agent Task 3: Contract Drafting Suggestions</a><br>
   6.4 <a href="#MultiStepWorkflows">Multi-Step Agent Workflows</a><br>
   6.5 <a href="#BusinessScenarios">Business Scenarios</a><br>

<a id='Dependencies'></a>  
## 1. Dependencies
[Back to table of contents](#TOC)

This section initializes the necessary dependencies and sets environment variables for connecting to your Weaviate Cloud instance.

**If using Colab, add your keys between the quotation marks in the cell below**

In [2]:
# un this cell to load your keys in Colab
import os

# set environment variables
os.environ['WEAVIATE_URL'] = ''  # weaviate instance url
os.environ['WEAVIATE_API_KEY'] = ''  # admin api key

**If running locally, make sure your keys are in your .env file and run this cell**

In [3]:
# Run this cell if you are working locally in VS code or Code Space
import dotenv

dotenv.load_dotenv(override=True)

True

<a id='Connecting'></a>  
## 2. Connecting to your Weaviate cluster  
[Back to table of contents](#TOC)  

To interact with our Weaviate cluster, we'll initialize a client object and verify the connection is successful. This connection will be used throughout the notebook for all data operations and Query Agent interactions.

In [4]:
import weaviate, os

# Connect to Weaviate Cloud
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.getenv("WEAVIATE_URL"),
    auth_credentials=weaviate.auth.AuthApiKey(os.getenv("WEAVIATE_API_KEY"))
)

# check if the connection is successful
client.is_ready()

True

<a id='Collections'></a>  
## 3. Setting up collections and loading data  
[Back to table of contents](#TOC)  

In this section, we'll create three collections (Brands, Ecommerce, and Books) and populate them with pre-vectorized data from the [Weaviate Agents dataset on Hugging Face](https://huggingface.co/datasets/weaviate/agents). Each collection uses the Snowflake Arctic Embed v2.0 model for consistent vector embeddings.

<a id='BrandsCollection'></a>  
### 3.1 Create and populate Brands collection  
[Back to table of contents](#TOC)  

We'll start by creating the Brands collection with properties for clothing brand information, then populate it with brand records from the Weaviate agents dataset.

In [5]:
# import required module for configuration
import weaviate.classes.config as wc

# delete collection if it exists
if client.collections.exists("Brands"):
    client.collections.delete("Brands")

client.collections.create(
    name="Brands", #Collection name

    #Set vectorizer configuration for text2vec using Weaviates embedding model
    vector_config=wc.Configure.Vectors.text2vec_weaviate(
        model="Snowflake/snowflake-arctic-embed-l-v2.0",
        source_properties=["name", "description", "child_brands", "parent_brand"],
    ),

)

print("Successfully created collection: Brands.")

Successfully created collection: Brands.


In [6]:
# Upload brands data using the Weaviate-recommended streaming approach
from datasets import load_dataset

# Load fresh streaming dataset for upload
brands_dataset = load_dataset("weaviate/agents", "query-agent-brands", split="train", streaming=True)

brands_collection = client.collections.get("Brands")
with brands_collection.batch.fixed_size(batch_size=100) as batch:
    for item in brands_dataset:
        batch.add_object(
            properties=item["properties"],
            vector=item["vector"]
        )

print("Successfully uploaded brands data using Weaviate streaming method!")

/Users/scottaskinosie/Documents/weaviate/weaviate_query_agent_demo/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Successfully uploaded brands data using Weaviate streaming method!


<a id='EcommerceCollection'></a>  
### 3.2 Create and populate Ecommerce collection  
[Back to table of contents](#TOC)  

Next, we'll create the Ecommerce collection for fashion items with properties including prices, categories, reviews, and brand associations, then load product records.

In [7]:
# import required module for configuration
import weaviate.classes.config as wc

# delete collection if it exists
if client.collections.exists("Ecommerce"):
    client.collections.delete("Ecommerce")

client.collections.create(
    name="Ecommerce", #Collection name

    #Set vectorizer configuration for text2vec using Weaviates embedding model
    vector_config=wc.Configure.Vectors.text2vec_weaviate(
        model="Snowflake/snowflake-arctic-embed-l-v2.0",
        source_properties=["name", "description", "collection", "category", "subcategory", "brand", "colors", "tags", "reviews"],
    ),
)

print("Successfully created collection: Ecommerce.")

Successfully created collection: Ecommerce.


In [8]:
# Upload ecommerce data using the Weaviate-recommended streaming approach
ecommerce_dataset = load_dataset("weaviate/agents", "query-agent-ecommerce", split="train", streaming=True)

ecommerce_collection = client.collections.get("Ecommerce")
with ecommerce_collection.batch.fixed_size(batch_size=100) as batch:
    for item in ecommerce_dataset:
        batch.add_object(
            properties=item["properties"],
            vector=item["vector"]
        )

print("Successfully uploaded ecommerce data using Weaviate streaming method!")

Successfully uploaded ecommerce data using Weaviate streaming method!


<a id='BooksCollection'></a>  
### 3.3 Create and populate Books collection  
[Back to table of contents](#TOC)  

Finally, we'll create the Books collection with properties for titles, authors, descriptions, and genres, then populate it with book records from various genres including mystery, fiction, and non-fiction.

In [9]:
# import required module for configuration
import weaviate.classes.config as wc

# delete collection if it exists
if client.collections.exists("Books"):
    client.collections.delete("Books")

client.collections.create(
    name="Books", #Collection name

    #Set vectorizer configuration for text2vec using Weaviates embedding model
    vector_config=wc.Configure.Vectors.text2vec_weaviate(
        model="Snowflake/snowflake-arctic-embed-l-v2.0",
        source_properties=["title", "author", "description", "genres"],
    ),
)

print("Successfully created collection: Books.")

Successfully created collection: Books.


In [10]:
# Upload books data using the Weaviate-recommended streaming approach
books_dataset = load_dataset("weaviate/agents", "query-agent-books", split="train", streaming=True)

books_collection = client.collections.get("Books")
with books_collection.batch.fixed_size(batch_size=100) as batch:
    for item in books_dataset:
        batch.add_object(
            properties=item["properties"],
            vector=item["vector"]
        )

print("Successfully uploaded books data using Weaviate streaming method!")

Successfully uploaded books data using Weaviate streaming method!


In [22]:
# import required module for configuration
import weaviate.classes.config as wc

# delete collection if it exists
if client.collections.exists("Financialcontracts"):
    client.collections.delete("Financialcontracts")

client.collections.create(
    name="Financialcontracts", #Collection name

    #Set vectorizer configuration for text2vec using Weaviates embedding model
    vector_config=wc.Configure.Vectors.text2vec_weaviate(
        model="Snowflake/snowflake-arctic-embed-l-v2.0",
        source_properties=["contract_text"],
    ),
)

print("Successfully created collection: Financialcontracts.")

Successfully created collection: Financialcontracts.


In [24]:
# Upload books data using the Weaviate-recommended streaming approach
books_dataset = load_dataset("weaviate/agents", "query-agent-financial-contracts", split="train", streaming=True)

books_collection = client.collections.get("Financialcontracts")
with books_collection.batch.fixed_size(batch_size=100) as batch:
    for item in books_dataset:
        batch.add_object(
            properties=item["properties"],
            vector=item["vector"]
        )

print("Successfully uploaded Financialcontracts data using Weaviate streaming method!")

Successfully uploaded Financialcontracts data using Weaviate streaming method!


<a id='QueryAgent'></a>  
## 4. Creating the Query Agent  
[Back to table of contents](#TOC)  

Now we'll create our Query Agent instance, which will intelligently route queries across our three collections (Brands, Ecommerce, Books) and automatically determine the optimal search strategy.

The output from the Query Agent will include: the original query, the generated answer to the query, the searches performed, the collections searched, filters applied, aggregates completed, source objects pulled from the database that comntributed to the generated answer and any mising information if the generated answer is incomplete.

In [25]:
from weaviate_agents.query import QueryAgent

# Instantiate a new agent object, and specify the collections to query
qa = QueryAgent(
    client=client,
    collections=["Brands", "Ecommerce", "Books", "Financialcontracts"],
)


<a id='ExampleQueries'></a>  
## 5. Example Query Agent interactions  
[Back to table of contents](#TOC)  

Let's explore the Query Agent's capabilities through various types of queries. Notice how the agent automatically determines which collections to search, applies filters, and performs aggregations based on the natural language input.

<a id='BookQueries'></a>  
### 5.0 Classic Vector Retrieval 
[Back to table of contents](#TOC)  

This Query Agent is able to perform classic vector database retrieval without initiating a subsequent generative step.

In [15]:
# Perform a vector search
response = qa.search(
    "Books about plants"
)

# Print the response
# The output will include: the original query, the generated answer to the query, the seraches performed,
# the collections searched, filters applied, aggregates completed and source objects ulled from the database.
print(response)

original_query='Books about plants' searches=[QueryResultWithCollection(queries=['plants'], filters=[[]], filter_operators='AND', collection='Books')] usage=Usage(requests=2, request_tokens=5118, response_tokens=59, total_tokens=5177, details={'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'cached_tokens': 0}) total_time=3.540576934814453 search_results=QueryReturn(objects=[Object(uuid=UUID('0210da9f-e653-4ceb-a938-a85245832767'), metadata={'creation_time': None, 'last_update_time': None, 'distance': None, 'certainty': None, 'score': 0.8203125, 'explain_score': None, 'is_consistent': None, 'rerank_score': None}, properties={'title': "The Botany of Desire: A Plant's-Eye View of the World", 'description': 'Every schoolchild learns about the mutually beneficial dance of honeybees and flowers: The bee collects nectar and pollen to make honey and, in the process, spreads the flowers’ genes far and wide. In The Botany of Desire, Mi

<a id='BookQueries'></a>  
### 5.1 Classic RAG search  
[Back to table of contents](#TOC)  

This Query Agent is able to respond as a classic RAG system. The Query Agent will run an initial generative step that will optimize the original query for a vector database.



In [16]:
# Perform a query
response = qa.run(
    "Are there any books about King Arthur or Knights that I should read?"
)

# Print the response
# The output will include: the original query, the generated answer to the query, the seraches performed,
# the collections searched, filters applied, aggregates completed and source objects ulled from the database.
response.display()

╭─────────────────────────────────────────────── 🔍 Original Query ───────────────────────────────────────────────╮
│                                                                                                                 │
│ Are there any books about King Arthur or Knights that I should read?                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📝 Final Answer ────────────────────────────────────────────────╮
│                                                                                                                 │
│ If you're interested in books about King Arthur and knights, here are a few highly recommended titles:          │
│                                                                                                                 │
│ 1. **The Winter King** by Bernard Cornwell – This is the first book in The Warlord Chronicles trilogy, offering │
│ a gripping, historically grounded retelling of Arthurian legend, filled with drama, war, and ancient magic.     │
│                                                                                                                 │
│ 2. **The Once and Future King** by T.H. White – A classic and beloved retelling of the Arthurian legends,       │
│ bringing the world of King Arthur and his court to life with humor, pathos, and unforgettable characters.       │
│                                                                                                                 │
│ 3. **A Knight of the Seven Kingdoms** by George R.R. Martin – While not about King Arthur specifically, this    │
│ collection of novellas set in the world of "A Song of Ice and Fire" follows the adventures of Ser Duncan the    │
│ Tall and his squire Egg. It's a fantastic exploration of knighthood, chivalry, and the lives of knights in a    │
│ medieval fantasy setting.                                                                                       │
│                                                                                                                 │
│ These books provide a mix of traditional Arthurian tales and fresh takes on the chivalric tradition, making     │
│ them great choices for fans of legendary kings and valiant knights.                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🔭 Searches Executed 1/2 ────────────────────────────────────────────╮
│                                                                                                                 │
│ QueryResultWithCollection(queries=['King Arthur'], filters=[[]], filter_operators='AND', collection='Books')    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🔭 Searches Executed 2/2 ────────────────────────────────────────────╮
│                                                                                                                 │
│ QueryResultWithCollection(                                                                                      │
│     queries=['books about knights'],                                                                            │
│     filters=[[]],                                                                                               │
│     filter_operators='AND',                                                                                     │
│     collection='Books'                                                                                          │
│ )                                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ 📊 No Aggregations Run                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── 📚 Sources ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  - object_id='8cd19ea9-9b92-47f2-9f5d-d081a1844a37' collection='Books'                                          │
│  - object_id='6c59ee0c-0f3e-4af9-96c1-e2965b0f9c70' collection='Books'                                          │
│  - object_id='7b202af9-2a60-4ae3-989d-f1720cedb78d' collection='Books'                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   📊 Usage Statistics    
┌────────────────┬───────┐
│ LLM Requests:  │ 5     │
│ Input Tokens:  │ 9814  │
│ Output Tokens: │ 350   │
│ Total Tokens:  │ 10164 │
└────────────────┴───────┘

Total Time Taken: 6.84s

<a id='BookQueries'></a>  
### 5.2 Generated database filtering   
[Back to table of contents](#TOC)  

This Query Agent is able determine appropriate filters and apply filtering for database objects returned by the vector database query. This allows for dynamic filtering in RAG systems, a step that is normally a manual process.


In [18]:
# Perform a query
response = qa.run(
    "I am in the mood for a good mystery book, what should I read?"
)

# Print the response
response.display()

╭─────────────────────────────────────────────── 🔍 Original Query ───────────────────────────────────────────────╮
│                                                                                                                 │
│ I am in the mood for a good mystery book, what should I read?                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📝 Final Answer ────────────────────────────────────────────────╮
│                                                                                                                 │
│ If you're in the mood for a great mystery, here are some excellent options to consider:                         │
│                                                                                                                 │
│ - The Keepsake by Tess Gerritsen: This suspenseful thriller features medical examiner Maura Isles and detective │
│ Jane Rizzoli, who investigate a series of chilling murders connected to ancient death rituals and a modern-day  │
│ killer stalking an archaeologist.                                                                               │
│                                                                                                                 │
│ - Cross Bones by Kathy Reichs: Forensic anthropologist Temperance Brennan is drawn into a case that links a     │
│ Montreal murder to secrets buried in the dust of Israel, involving ancient mysteries and explosive              │
│ consequences.                                                                                                   │
│                                                                                                                 │
│ - The Mysterious Affair at Styles by Agatha Christie: Agatha Christie’s classic introduces Hercule Poirot as he │
│ unravels a complex poisoning at an English country estate. It's a perfect choice if you enjoy traditional       │
│ detective stories with clever twists.                                                                           │
│                                                                                                                 │
│ - The Last Book by Zoran Živković: A literary mystery involving mysterious deaths at a bookstore, a secret      │
│ apocalyptic sect, and a race against time to discover the truth behind an elusive, possibly deadly, book.       │
│                                                                                                                 │
│ Choose based on your mood—whether you want forensic intrigue, classic sleuthing, or a twist of the supernatural │
│ in your mystery read.                                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🔭 Searches Executed 1/1 ────────────────────────────────────────────╮
│                                                                                                                 │
│ QueryResultWithCollection(                                                                                      │
│     queries=['mystery books'],                                                                                  │
│     filters=[                                                                                                   │
│         [                                                                                                       │
│             TextArrayPropertyFilter(                                                                            │
│                 property_name='genres',                                                                         │
│                 operator=<ComparisonOperator.CONTAINS_ANY: 'contains_any'>,                                     │
│                 value=['Mystery']                                                                               │
│             )                                                                                                   │
│         ]                                                                                                       │
│     ],                                                                                                          │
│     filter_operators='AND',                                                                                     │
│     collection='Books'                                                                                          │
│ )                                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ 📊 No Aggregations Run                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── 📚 Sources ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  - object_id='55220a5f-0fae-487e-b524-dfe1a85b9044' collection='Books'                                          │
│  - object_id='32afd7da-bca3-432a-85ca-db2ee6e9c0a8' collection='Books'                                          │
│  - object_id='2679a1a3-56c9-4ee6-9592-d8040feef845' collection='Books'                                          │
│  - object_id='6d31192f-e02e-4ecf-8fb8-1a12205b2a11' collection='Books'                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   📊 Usage Statistics   
┌────────────────┬──────┐
│ LLM Requests:  │ 4    │
│ Input Tokens:  │ 9274 │
│ Output Tokens: │ 338  │
│ Total Tokens:  │ 9612 │
└────────────────┴──────┘

Total Time Taken: 8.13s

<a id='StatisticalQueries'></a>  
### 5.3 Statistical and aggregation queries  
[Back to table of contents](#TOC)  

The Query Agent automatically recognizes when aggregation is needed and performs complex statistical operations like counting and grouping data.

In [19]:
# Perform a query
response = qa.run(
    "Which author has the most books in my collection?"
)

# Print the response
response.display()

╭─────────────────────────────────────────────── 🔍 Original Query ───────────────────────────────────────────────╮
│                                                                                                                 │
│ Which author has the most books in my collection?                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📝 Final Answer ────────────────────────────────────────────────╮
│                                                                                                                 │
│ The author with the most books in your collection is Stephen King, with a total of 57 books.                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ 🔭 No Searches Run                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 📊 Aggregations Run 1/1 ────────────────────────────────────────────╮
│                                                                                                                 │
│ AggregationResultWithCollection(                                                                                │
│     search_query=None,                                                                                          │
│     groupby_property='author',                                                                                  │
│     aggregations=[                                                                                              │
│         TextPropertyAggregation(                                                                                │
│             property_name='title',                                                                              │
│             metrics=<TextMetrics.COUNT: 'COUNT'>,                                                               │
│             top_occurrences_limit=None                                                                          │
│         )                                                                                                       │
│     ],                                                                                                          │
│     filters=[],                                                                                                 │
│     collection='Books'                                                                                          │
│ )                                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   📊 Usage Statistics    
┌────────────────┬───────┐
│ LLM Requests:  │ 4     │
│ Input Tokens:  │ 62232 │
│ Output Tokens: │ 124   │
│ Total Tokens:  │ 62356 │
└────────────────┴───────┘

Total Time Taken: 7.62s

<a id='MultipleQueries'></a>  
### 5.4 Querying multiple databases  
[Back to table of contents](#TOC)  

The Query agent looks at collection descriptions and property data to determine which collection or collections to query for proper context for the query response generation.

In [20]:
# Perform a query
response = qa.run(
    """I am looking to buy a cool new jean jacket or light coat at or below $150.
    I want a trendy but also timeless look but can also hold up and last me a long time.
    If possible, I want to purchase something from a new company that might get me a
    sponsorship for my social media account."""
)
# Print the response
response.display()

╭─────────────────────────────────────────────── 🔍 Original Query ───────────────────────────────────────────────╮
│                                                                                                                 │
│ I am looking to buy a cool new jean jacket or light coat at or below $150.                                      │
│     I want a trendy but also timeless look but can also hold up and last me a long time.                        │
│     If possible, I want to purchase something from a new company that might get me a                            │
│     sponsorship for my social media account.                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📝 Final Answer ────────────────────────────────────────────────╮
│                                                                                                                 │
│ Here are two standout options for a trendy, timeless, and durable coat under $150, both from emerging brands    │
│ that could offer future sponsorship potential:                                                                  │
│                                                                                                                 │
│ 1. **Ivory Parchment Trench by Canvas & Co.**                                                                   │
│    - Price: $150                                                                                                │
│    - Style: Elegant, tailored trench coat in a fresh ivory shade, inspired by classical art for a timeless      │
│ appeal.                                                                                                         │
│    - Features: Tailored lines and decorative button details, perfect for both trendy looks and lasting style.   │
│    - Brand: Canvas & Co., under Canvas Cosmos (founded 2018, known for quality craftsmanship and timeless       │
│ design, and still relatively new in the fashion scene).                                                         │
│    - Feedback: Customers rave about its elegance and the compliments they receive.                              │
│                                                                                                                 │
│ 2. **Elysian Quill Coat by Loom & Aura**                                                                        │
│    - Price: $150                                                                                                │
│    - Style: Vintage-inspired with a refined silhouette; intricate buttons add classic yet trendy flair.         │
│    - Features: Available in white or grey; ideal for making a sophisticated, lasting fashion statement.         │
│    - Brand: Loom & Aura, part of Loom Legacy (founded 2020, praised for timeless elegance and longevity).       │
│    - Feedback: Reviewers highlight its comfort, unique style, and durability.                                   │
│                                                                                                                 │
│ Both Canvas & Co. and Loom & Aura come from emerging brands established in the last several years. These brands │
│ value craftsmanship and timelessness—qualities that resonate with both trendsetting and enduring style.         │
│ Featuring their products and reviews on your social media could make you an attractive candidate for a brand    │
│ partnership, given their contemporary vision and growth phase.                                                  │
│                                                                                                                 │
│ As for jean jackets under $150, specific examples weren't provided, but reviews indicate that jackets in this   │
│ range are often praised for their lasting quality, unique designs, and fit. For even greater sponsorship        │
│ potential, consider reaching out directly to the brand's PR or marketing teams expressing your interest in      │
│ collaboration and sharing your social media presence.                                                           │
│                                                                                                                 │
│ If you’d like more options, particularly for classic jean jackets, or want contact recommendations for          │
│ arranging potential partnerships, let me know!                                                                  │
│                                                                                                                 │
╰────────────────────────────────────────────────────────

╭─────────────────────────────────────────── 🔭 Searches Executed 1/2 ────────────────────────────────────────────╮
│                                                                                                                 │
│ QueryResultWithCollection(                                                                                      │
│     queries=['trendy timeless jean jacket', 'trendy timeless light coat'],                                      │
│     filters=[                                                                                                   │
│         [                                                                                                       │
│             IntegerPropertyFilter(                                                                              │
│                 property_name='price',                                                                          │
│                 operator=<ComparisonOperator.LESS_EQUAL: '<='>,                                                 │
│                 value=150.0                                                                                     │
│             ),                                                                                                  │
│             TextPropertyFilter(                                                                                 │
│                 property_name='subcategory',                                                                    │
│                 operator=<ComparisonOperator.LIKE: 'LIKE'>,                                                     │
│                 value='*jean jacket*'                                                                           │
│             )                                                                                                   │
│         ],                                                                                                      │
│         [                                                                                                       │
│             IntegerPropertyFilter(                                                                              │
│                 property_name='price',                                                                          │
│                 operator=<ComparisonOperator.LESS_EQUAL: '<='>,                                                 │
│                 value=150.0                                                                                     │
│             ),                                                                                                  │
│             TextPropertyFilter(                                                                                 │
│                 property_name='subcategory',                                                                    │
│                 operator=<ComparisonOperator.LIKE: 'LIKE'>,                                                     │
│                 value='*coat*'                                                                                  │
│             )                                                                                                   │
│         ]                                                                                                       │
│     ],                                                                                                          │
│     filter_operators='AND',                                                                                     │
│     collection='Ecommerce'                                                                                      │
│ )                                                                                                               │
│                                                                                                                 │
╰────────────────────────────────────────────────────────

╭─────────────────────────────────────────── 🔭 Searches Executed 2/2 ────────────────────────────────────────────╮
│                                                                                                                 │
│ QueryResultWithCollection(                                                                                      │
│     queries=[                                                                                                   │
│         'emerging fashion brands suitable for social media sponsorship',                                        │
│         'new clothing brands open to collaborations'                                                            │
│     ],                                                                                                          │
│     filters=[                                                                                                   │
│         [                                                                                                       │
│             IntegerPropertyFilter(                                                                              │
│                 property_name='foundation_year',                                                                │
│                 operator=<ComparisonOperator.GREATER_EQUAL: '>='>,                                              │
│                 value=2018.0                                                                                    │
│             )                                                                                                   │
│         ],                                                                                                      │
│         [                                                                                                       │
│             IntegerPropertyFilter(                                                                              │
│                 property_name='foundation_year',                                                                │
│                 operator=<ComparisonOperator.GREATER_EQUAL: '>='>,                                              │
│                 value=2018.0                                                                                    │
│             )                                                                                                   │
│         ]                                                                                                       │
│     ],                                                                                                          │
│     filter_operators='AND',                                                                                     │
│     collection='Brands'                                                                                         │
│ )                                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 📊 Aggregations Run 1/2 ────────────────────────────────────────────╮
│                                                                                                                 │
│ AggregationResultWithCollection(                                                                                │
│     search_query=None,                                                                                          │
│     groupby_property='subcategory',                                                                             │
│     aggregations=[                                                                                              │
│         TextPropertyAggregation(                                                                                │
│             property_name='reviews',                                                                            │
│             metrics=<TextMetrics.COUNT: 'COUNT'>,                                                               │
│             top_occurrences_limit=None                                                                          │
│         ),                                                                                                      │
│         TextPropertyAggregation(                                                                                │
│             property_name='reviews',                                                                            │
│             metrics=<TextMetrics.TOP_OCCURRENCES: 'TOP_OCCURRENCES'>,                                           │
│             top_occurrences_limit=5                                                                             │
│         )                                                                                                       │
│     ],                                                                                                          │
│     filters=[                                                                                                   │
│         TextPropertyFilter(                                                                                     │
│             property_name='subcategory',                                                                        │
│             operator=<ComparisonOperator.LIKE: 'LIKE'>,                                                         │
│             value='*Jean Jacket*'                                                                               │
│         ),                                                                                                      │
│         TextPropertyFilter(                                                                                     │
│             property_name='subcategory',                                                                        │
│             operator=<ComparisonOperator.LIKE: 'LIKE'>,                                                         │
│             value='*Light Coat*'                                                                                │
│         ),                                                                                                      │
│         IntegerPropertyFilter(                                                                                  │
│             property_name='price',                                                                              │
│             operator=<ComparisonOperator.LESS_THAN: '<'>,                                                       │
│             value=150.0                                                                                         │
│         )                                                                                                       │
│     ],                                                                                                          │
│     collection='Ecommerce'                             

╭──────────────────────────────────────────── 📊 Aggregations Run 2/2 ────────────────────────────────────────────╮
│                                                                                                                 │
│ AggregationResultWithCollection(                                                                                │
│     search_query=None,                                                                                          │
│     groupby_property=None,                                                                                      │
│     aggregations=[                                                                                              │
│         IntegerPropertyAggregation(property_name='foundation_year', metrics=<NumericMetrics.COUNT: 'COUNT'>)    │
│     ],                                                                                                          │
│     filters=[                                                                                                   │
│         IntegerPropertyFilter(                                                                                  │
│             property_name='foundation_year',                                                                    │
│             operator=<ComparisonOperator.GREATER_EQUAL: '>='>,                                                  │
│             value=2020.0                                                                                        │
│         )                                                                                                       │
│     ],                                                                                                          │
│     collection='Brands'                                                                                         │
│ )                                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────── ⚠️ Answer is Partial - Missing Information: ───────────────────────────────────╮
│                                                                                                                 │
│ - The user asked specifically for options including jean jackets under $150 which was not provided in the       │
│ answer.                                                                                                         │
│ - Specific details about how to contact the brands for sponsorship or examples of how to engage with the brands │
│ for social media sponsorship are missing.                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── 📚 Sources ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  - object_id='e9da5341-26de-4eb5-b0ab-b1985ab8ae47' collection='Ecommerce'                                      │
│  - object_id='f3ced7f1-810f-40bb-b832-dcb8c571231b' collection='Ecommerce'                                      │
│  - object_id='535a9a67-cd0a-446f-8dc5-d270c1c690e4' collection='Brands'                                         │
│  - object_id='9ca3a3dd-c1fe-439b-a977-3df748ef6b3d' collection='Brands'                                         │
│  - object_id='bd55f1b1-63e5-4940-bf90-9a0bb40798b2' collection='Brands'                                         │
│  - object_id='e0a96366-8514-45af-b403-2d1913993cba' collection='Brands'                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   📊 Usage Statistics    
┌────────────────┬───────┐
│ LLM Requests:  │ 7     │
│ Input Tokens:  │ 31538 │
│ Output Tokens: │ 940   │
│ Total Tokens:  │ 32478 │
└────────────────┴───────┘

Total Time Taken: 17.81s

<a id='MissingInfo'></a>  
### 5.5 Identifying missing information  
[Back to table of contents](#TOC)  

If a partial response is generated for a query the Query Agent will report the missing information not present in the database.

In [21]:
# Perform a query
response = qa.run(
    """I want to buy a new leather jacket but I am vegan and only buy from socially responsible brands."""
)
# Print the response
response.display()

╭─────────────────────────────────────────────── 🔍 Original Query ───────────────────────────────────────────────╮
│                                                                                                                 │
│ I want to buy a new leather jacket but I am vegan and only buy from socially responsible brands.                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📝 Final Answer ────────────────────────────────────────────────╮
│                                                                                                                 │
│ There are a few options for vegan jackets from potentially socially responsible brands. While there isn’t       │
│ explicit confirmation here that these brands are fully vegan or have detailed social responsibility             │
│ certifications, Loom & Aura stands out as a promising choice, and their jackets do not use leather:             │
│                                                                                                                 │
│ - The Shroud Jacket by Vivid Verse is an ultra-resistant fabric jacket with a cyberpunk aesthetic, rather than  │
│ leather or faux leather. It’s described as protective and stylish, with metallic trims.                         │
│ - Loom & Aura offers the Garden Dusk Linen Jacket (soft linen, subtle lavender color, $90) and the Mystic       │
│ Forest Velvet Cloak (luxurious velvet, embroidered woodland patterns, $120). Both are animal-free alternatives. │
│                                                                                                                 │
│ Loom & Aura is mentioned among brands, and their product materials appear to align with vegan principles        │
│ (linen, velvet). However, there’s no direct mention of their full ethical or social responsibility policies in  │
│ the available information.                                                                                      │
│                                                                                                                 │
│ If vegan material and animal-free design are your primary concern, these jackets from Loom & Aura, in           │
│ particular, are suitable alternatives to leather. For full assurance of both vegan and strong social            │
│ responsibility standards, further brand research or direct confirmation is recommended, as that detailed data   │
│ isn’t provided here.                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🔭 Searches Executed 1/2 ────────────────────────────────────────────╮
│                                                                                                                 │
│ QueryResultWithCollection(                                                                                      │
│     queries=['vegan alternative leather jacket', 'vegan faux leather jacket'],                                  │
│     filters=[[], []],                                                                                           │
│     filter_operators='OR',                                                                                      │
│     collection='Ecommerce'                                                                                      │
│ )                                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🔭 Searches Executed 2/2 ────────────────────────────────────────────╮
│                                                                                                                 │
│ QueryResultWithCollection(                                                                                      │
│     queries=[                                                                                                   │
│         'socially responsible brands',                                                                          │
│         'socially conscious brands',                                                                            │
│         'ethical brands',                                                                                       │
│         'sustainable brands'                                                                                    │
│     ],                                                                                                          │
│     filters=[[], [], [], []],                                                                                   │
│     filter_operators='OR',                                                                                      │
│     collection='Brands'                                                                                         │
│ )                                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ 📊 No Aggregations Run                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────── ⚠️ Answer is Partial - Missing Information: ───────────────────────────────────╮
│                                                                                                                 │
│ - Confirmation or details on the social responsibility or ethical policies of the brands mentioned (Vivid       │
│ Verse, Loom & Aura) specifically regarding vegan practices and social responsibility is missing. The answer     │
│ acknowledges this gap but does not provide direct brand-level social responsibility or ethical certification    │
│ information.                                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── 📚 Sources ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  - object_id='01e6a678-77b9-4193-9cae-518cfed2f364' collection='Ecommerce'                                      │
│  - object_id='07f586c9-d0d5-445a-9f42-bed957358d1a' collection='Ecommerce'                                      │
│  - object_id='d92af99c-aef2-4228-9917-b97340127274' collection='Ecommerce'                                      │
│  - object_id='e3ec0ad0-4502-4946-9f98-77cd8db01c8e' collection='Brands'                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   📊 Usage Statistics    
┌────────────────┬───────┐
│ LLM Requests:  │ 5     │
│ Input Tokens:  │ 11062 │
│ Output Tokens: │ 459   │
│ Total Tokens:  │ 11521 │
└────────────────┴───────┘

Total Time Taken: 8.31s

<a id='ContractAnalysis'></a>  
## 6. Practical Business Use Case for the Query Agent: Contract Analysis  
[Back to table of contents](#TOC)  

This section demonstrates practical business applications of the Query Agent through contract analysis scenarios. We'll explore how the agent can be used for risk assessment, contract comparison, and drafting suggestions in real-world business contexts.

<a id='AgentTask1'></a>  
### 6.1 Agent Task 1: Risk Assessment  
[Back to table of contents](#TOC)

In [26]:
# Perform a query
response = qa.run(
    "What are the top 3 risks in employment contracts that employers should watch out for? Provide specific examples."
)

# Print the response
response.display()

╭─────────────────────────────────────────────── 🔍 Original Query ───────────────────────────────────────────────╮
│                                                                                                                 │
│ What are the top 3 risks in employment contracts that employers should watch out for? Provide specific          │
│ examples.                                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📝 Final Answer ────────────────────────────────────────────────╮
│                                                                                                                 │
│ The top three risks in employment contracts that employers should watch out for are:                            │
│                                                                                                                 │
│ 1. **Unclear or Inadequate Termination Clauses**                                                                │
│    - Risk: If the termination provisions are not clear or specific, employers may face legal challenges or be   │
│ required to provide longer notice periods or greater compensation than intended. For example, if a contract     │
│ simply allows for termination "at any time" but does not detail grounds for immediate termination or specify    │
│ procedures, this can cause disputes if an employee is let go without cause.                                     │
│    - Example: A contract stating, "Either party may terminate at any time with two weeks’ notice," (as seen in  │
│ the provided contracts) could leave the employer exposed if state law or company policy requires a different    │
│ procedure, or if "cause" for immediate termination is not clearly defined.                                      │
│                                                                                                                 │
│ 2. **Ambiguous Confidentiality or Non-Disclosure Provisions**                                                   │
│    - Risk: If confidentiality clauses are not specific or enforceable, employees may disclose proprietary       │
│ information either during or after employment, causing significant harm to the company.                         │
│    - Example: Simply including a clause like, "The Employee agrees to maintain confidentiality of all           │
│ proprietary information during and after the term of employment," without defining what constitutes             │
│ "proprietary information" can lead to disputes and make enforcement difficult.                                  │
│                                                                                                                 │
│ 3. **Vague Definitions of Duties and Performance Expectations**                                                 │
│    - Risk: If an employment contract does not clearly outline the employee's roles, responsibilities, or        │
│ performance metrics, it can lead to disagreements about what is expected of the employee, difficulties in       │
│ performance management, and possible wrongful termination claims.                                               │
│    - Example: Contracts that state, "The Employee agrees to perform the duties and responsibilities as assigned │
│ by the Employer," without listing those duties, open the door for misunderstandings about job expectations.     │
│                                                                                                                 │
│ Employers should ensure all contract terms—especially those related to termination, confidentiality, and        │
│ duties—are written in clear, precise language to minimize these risks.                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🔭 Searches Executed 1/2 ────────────────────────────────────────────╮
│                                                                                                                 │
│ QueryResultWithCollection(                                                                                      │
│     queries=['common risks in employment contracts for employers'],                                             │
│     filters=[                                                                                                   │
│         [                                                                                                       │
│             TextPropertyFilter(                                                                                 │
│                 property_name='contract_type',                                                                  │
│                 operator=<ComparisonOperator.LIKE: 'LIKE'>,                                                     │
│                 value='*employment*'                                                                            │
│             )                                                                                                   │
│         ]                                                                                                       │
│     ],                                                                                                          │
│     filter_operators='AND',                                                                                     │
│     collection='Financialcontracts'                                                                             │
│ )                                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🔭 Searches Executed 2/2 ────────────────────────────────────────────╮
│                                                                                                                 │
│ QueryResultWithCollection(                                                                                      │
│     queries=['employment contract risks for employers with example'],                                           │
│     filters=[                                                                                                   │
│         [                                                                                                       │
│             TextPropertyFilter(                                                                                 │
│                 property_name='contract_type',                                                                  │
│                 operator=<ComparisonOperator.LIKE: 'LIKE'>,                                                     │
│                 value='*employment*'                                                                            │
│             )                                                                                                   │
│         ]                                                                                                       │
│     ],                                                                                                          │
│     filter_operators='AND',                                                                                     │
│     collection='Financialcontracts'                                                                             │
│ )                                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ 📊 No Aggregations Run                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────── ⚠️ Answer is Partial - Missing Information: ───────────────────────────────────╮
│                                                                                                                 │
│ - The answer did not include specific examples of risks related to benefits, compensation disputes or governing │
│ law considerations, though such clauses appear in the contracts.                                                │
│ - There was no mention of risks arising from ambiguous bonus and benefit terms, which are present in multiple   │
│ contracts.                                                                                                      │
│ - The answer lacks insight about the risks in contracts related to employment term length and renewal           │
│ provisions, which could impact employer obligations.                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── 📚 Sources ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  - object_id='3d078c02-09f3-4aec-9c89-3b0b3fada95b' collection='Financialcontracts'                             │
│  - object_id='eca9770a-c3ca-4cff-8180-bf35537da90a' collection='Financialcontracts'                             │
│  - object_id='6731310b-a9a1-4e13-a954-cb7bf246b970' collection='Financialcontracts'                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   📊 Usage Statistics    
┌────────────────┬───────┐
│ LLM Requests:  │ 5     │
│ Input Tokens:  │ 14848 │
│ Output Tokens: │ 631   │
│ Total Tokens:  │ 15479 │
└────────────────┴───────┘

Total Time Taken: 15.89s

In [27]:
# Perform a query
response = qa.run(
    "Find contracts with payment terms and analyze what payment-related risks the paying party might face."
)

# Print the response
response.display()

╭─────────────────────────────────────────────── 🔍 Original Query ───────────────────────────────────────────────╮
│                                                                                                                 │
│ Find contracts with payment terms and analyze what payment-related risks the paying party might face.           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📝 Final Answer ────────────────────────────────────────────────╮
│                                                                                                                 │
│ Here is an analysis based on several contracts containing payment terms:                                        │
│                                                                                                                 │
│ 1. **Purchase Orders & Invoices**:                                                                              │
│    - Payment is typically due within 30 days of receiving an invoice.                                           │
│    - Some contracts include late fee provisions (e.g., a $50 late fee for payments overdue past 30 days).       │
│    - Payment must often be made by transferring funds to a specified bank account.                              │
│                                                                                                                 │
│ 2. **Loan Agreements**:                                                                                         │
│    - Repayment schedules can involve monthly installments over several years, with a specified interest rate.   │
│    - Contracts may stipulate that if the borrower defaults (fails to pay on schedule), the entire balance       │
│ becomes immediately due, and additional collection costs (including attorney's fees) can be charged.            │
│                                                                                                                 │
│ **Payment-Related Risks for the Paying Party**:                                                                 │
│ - **Disputed Goods or Services**: If goods or services are not delivered as agreed, the payer may still be      │
│ obligated to pay within the 30-day window, leading to risks if disputes arise after payment.                    │
│ - **Late Fees and Penalties**: Missing the payment deadline can result in additional fees or penalties,         │
│ increasing the total amount due.                                                                                │
│ - **Ambiguous Delivery or Invoice Dates**: If contract language is unclear about when the invoice is            │
│ "received," this can create confusion over payment deadlines and risk of accidental late payments.              │
│ - **Banking Errors or Fraud**: If the payment instructions are not carefully checked, there is a risk of funds  │
│ being transferred to the wrong account.                                                                         │
│ - **Defaults in Loan Agreements**: In a loan scenario, failing to adhere to the repayment schedule can trigger  │
│ immediate acceleration of the debt and collection costs.                                                        │
│ - **Legal Jurisdiction**: Contracts often specify a governing law, and payment disputes may have to be resolved │
│ in an unfamiliar legal environment, increasing costs and complexity.                                            │
│ - **Unverified Contract Changes**: Changes to payment terms not formally agreed upon can expose the paying      │
│ party to disputes or unexpected obligations.                                                                    │
│                                                                                                                 │
│ Overall, the key payment-related risks for the paying party are late fees, potential disputes over delivery or  │
│ invoicing, procedural or banking errors, and increased liability in the event of default or legal action.       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🔭 Searches Executed 1/2 ────────────────────────────────────────────╮
│                                                                                                                 │
│ QueryResultWithCollection(                                                                                      │
│     queries=['payment terms'],                                                                                  │
│     filters=[[]],                                                                                               │
│     filter_operators='AND',                                                                                     │
│     collection='Financialcontracts'                                                                             │
│ )                                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🔭 Searches Executed 2/2 ────────────────────────────────────────────╮
│                                                                                                                 │
│ QueryResultWithCollection(                                                                                      │
│     queries=['payment terms payment-related risks', 'payment risk paying party contract'],                      │
│     filters=[                                                                                                   │
│         [                                                                                                       │
│             TextPropertyFilter(                                                                                 │
│                 property_name='contract_text',                                                                  │
│                 operator=<ComparisonOperator.LIKE: 'LIKE'>,                                                     │
│                 value='*payment term*'                                                                          │
│             )                                                                                                   │
│         ],                                                                                                      │
│         [                                                                                                       │
│             TextPropertyFilter(                                                                                 │
│                 property_name='contract_text',                                                                  │
│                 operator=<ComparisonOperator.LIKE: 'LIKE'>,                                                     │
│                 value='*payment*'                                                                               │
│             )                                                                                                   │
│         ]                                                                                                       │
│     ],                                                                                                          │
│     filter_operators='AND',                                                                                     │
│     collection='Financialcontracts'                                                                             │
│ )                                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ 📊 No Aggregations Run                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── 📚 Sources ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  - object_id='54b68792-827c-44dd-b71d-bb1d65d5d151' collection='Financialcontracts'                             │
│  - object_id='c36ae346-0854-4cae-96c5-6effd480a908' collection='Financialcontracts'                             │
│  - object_id='741ce4e2-93d3-40d0-9712-3d0532ff3ba0' collection='Financialcontracts'                             │
│  - object_id='8dcc4d18-4679-496d-92de-30e2e84c1b2d' collection='Financialcontracts'                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   📊 Usage Statistics    
┌────────────────┬───────┐
│ LLM Requests:  │ 5     │
│ Input Tokens:  │ 11481 │
│ Output Tokens: │ 597   │
│ Total Tokens:  │ 12078 │
└────────────────┴───────┘

Total Time Taken: 11.73s

<a id='AgentTask2'></a>  
### 6.2 Agent Task 2: Contract Comparison  
[Back to table of contents](#TOC)

In [29]:
# Perform a query
response = qa.run(
    "Find 2 similar service agreements and compare their termination clauses. Which one is more favorable for the service provider?"
)

# Print the response
response.display()

╭─────────────────────────────────────────────── 🔍 Original Query ───────────────────────────────────────────────╮
│                                                                                                                 │
│ Find 2 similar service agreements and compare their termination clauses. Which one is more favorable for the    │
│ service provider?                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📝 Final Answer ────────────────────────────────────────────────╮
│                                                                                                                 │
│ Here is a comparison of two similar service agreements and their termination clauses:                           │
│                                                                                                                 │
│ **Agreement 1:**                                                                                                │
│ - Parties: Weaviate (Client) and Danny Williams (Service Provider)                                              │
│ - Main Services: Digital marketing services                                                                     │
│ - Termination Clause: Either party may terminate the agreement with thirty (30) days written notice to the      │
│ other party, but only in the event of a breach of any material term of the agreement.                           │
│                                                                                                                 │
│ **Agreement 2:**                                                                                                │
│ - Parties: Weaviate (Service Provider) and Danny Williams (Client)                                              │
│ - Main Services: AI development and consulting services                                                         │
│ - Termination Clause: Either party may terminate the agreement by providing thirty (30) days written notice to  │
│ the other party, with no requirement for breach or cause.                                                       │
│                                                                                                                 │
│ **Comparison:**                                                                                                 │
│ - Agreement 1 restricts termination to situations where a material breach has occurred, making it less flexible │
│ to exit the contract for any other reason.                                                                      │
│ - Agreement 2 allows either party to terminate for any reason, as long as they provide 30 days’ notice.         │
│                                                                                                                 │
│ **Which is more favorable for the service provider?**                                                           │
│ Agreement 2 is more favorable for the service provider because it gives them the right to terminate the         │
│ contract for any reason (or no reason at all) with only 30 days’ notice, providing maximum flexibility and less │
│ risk of being locked into an unfavorable arrangement. In contrast, Agreement 1 only allows termination in the   │
│ case of breach, which restricts the service provider’s ability to exit the agreement.                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🔭 Searches Executed 1/3 ────────────────────────────────────────────╮
│                                                                                                                 │
│ QueryResultWithCollection(                                                                                      │
│     queries=['service agreement contract'],                                                                     │
│     filters=[                                                                                                   │
│         [                                                                                                       │
│             TextPropertyFilter(                                                                                 │
│                 property_name='contract_type',                                                                  │
│                 operator=<ComparisonOperator.LIKE: 'LIKE'>,                                                     │
│                 value='*service agreement*'                                                                     │
│             )                                                                                                   │
│         ]                                                                                                       │
│     ],                                                                                                          │
│     filter_operators='AND',                                                                                     │
│     collection='Financialcontracts'                                                                             │
│ )                                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🔭 Searches Executed 2/3 ────────────────────────────────────────────╮
│                                                                                                                 │
│ QueryResultWithCollection(                                                                                      │
│     queries=['service agreement termination clause'],                                                           │
│     filters=[                                                                                                   │
│         [                                                                                                       │
│             TextPropertyFilter(                                                                                 │
│                 property_name='contract_type',                                                                  │
│                 operator=<ComparisonOperator.LIKE: 'LIKE'>,                                                     │
│                 value='*service agreement*'                                                                     │
│             )                                                                                                   │
│         ]                                                                                                       │
│     ],                                                                                                          │
│     filter_operators='AND',                                                                                     │
│     collection='Financialcontracts'                                                                             │
│ )                                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🔭 Searches Executed 3/3 ────────────────────────────────────────────╮
│                                                                                                                 │
│ QueryResultWithCollection(                                                                                      │
│     queries=['service agreement'],                                                                              │
│     filters=[                                                                                                   │
│         [                                                                                                       │
│             TextPropertyFilter(                                                                                 │
│                 property_name='contract_type',                                                                  │
│                 operator=<ComparisonOperator.EQUALS: '='>,                                                      │
│                 value='service agreement'                                                                       │
│             )                                                                                                   │
│         ]                                                                                                       │
│     ],                                                                                                          │
│     filter_operators='AND',                                                                                     │
│     collection='Financialcontracts'                                                                             │
│ )                                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ 📊 No Aggregations Run                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── 📚 Sources ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  - object_id='717880cc-4aeb-4032-9041-e173cc20d88a' collection='Financialcontracts'                             │
│  - object_id='285d27af-c55c-4952-ac25-dfa0085999f6' collection='Financialcontracts'                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   📊 Usage Statistics    
┌────────────────┬───────┐
│ LLM Requests:  │ 6     │
│ Input Tokens:  │ 19535 │
│ Output Tokens: │ 516   │
│ Total Tokens:  │ 20051 │
└────────────────┴───────┘

Total Time Taken: 12.61s

<a id='AgentTask3'></a>  
### 6.3 Agent Task 3: Contract Drafting Suggestions  
[Back to table of contents](#TOC)

In [30]:
# Perform a query
response = qa.run(
    "Based on the contracts in the database, what are 5 essential clauses that every service agreement should include to protect both parties?"
)

# Print the response
response.display()

╭─────────────────────────────────────────────── 🔍 Original Query ───────────────────────────────────────────────╮
│                                                                                                                 │
│ Based on the contracts in the database, what are 5 essential clauses that every service agreement should        │
│ include to protect both parties?                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📝 Final Answer ────────────────────────────────────────────────╮
│                                                                                                                 │
│ Based on the contracts in the database, five essential clauses that every service agreement should include to   │
│ protect both parties are:                                                                                       │
│                                                                                                                 │
│ 1. **Services Provided**: Clearly define the scope and nature of the services to be rendered. This ensures both │
│ parties have a mutual understanding of what will be delivered.                                                  │
│                                                                                                                 │
│ 2. **Compensation/Payment Terms**: Specify the total fees, payment schedule, and any installment arrangements.  │
│ This protects both parties by outlining financial obligations and preventing payment disputes.                  │
│                                                                                                                 │
│ 3. **Term and Duration**: State the length of the agreement and the start and end dates. This provides clarity  │
│ on how long the agreement is in effect.                                                                         │
│                                                                                                                 │
│ 4. **Termination Clause**: Allow either party to terminate the agreement by providing written notice, often 30  │
│ days. This gives both sides a structured way to exit the agreement if needed.                                   │
│                                                                                                                 │
│ 5. **Confidentiality**: Require both parties to maintain the confidentiality of any proprietary or sensitive    │
│ information exchanged during the agreement. This protects intellectual property and business interests.         │
│                                                                                                                 │
│ Including these clauses helps ensure clarity, manage expectations, and reduce the risk of misunderstandings or  │
│ disputes for both parties involved in a service agreement.                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🔭 Searches Executed 1/3 ────────────────────────────────────────────╮
│                                                                                                                 │
│ QueryResultWithCollection(                                                                                      │
│     queries=['essential clauses in service agreement'],                                                         │
│     filters=[                                                                                                   │
│         [                                                                                                       │
│             TextPropertyFilter(                                                                                 │
│                 property_name='contract_type',                                                                  │
│                 operator=<ComparisonOperator.EQUALS: '='>,                                                      │
│                 value='service agreement'                                                                       │
│             )                                                                                                   │
│         ]                                                                                                       │
│     ],                                                                                                          │
│     filter_operators='AND',                                                                                     │
│     collection='Financialcontracts'                                                                             │
│ )                                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🔭 Searches Executed 2/3 ────────────────────────────────────────────╮
│                                                                                                                 │
│ QueryResultWithCollection(                                                                                      │
│     queries=[                                                                                                   │
│         'essential protective clauses provisions service agreement',                                            │
│         'important clauses to protect both parties service contract',                                           │
│         'key provisions service agreement protective both parties',                                             │
│         'must-have protective service contract clauses',                                                        │
│         'service agreement critical protection provisions'                                                      │
│     ],                                                                                                          │
│     filters=[                                                                                                   │
│         [                                                                                                       │
│             TextPropertyFilter(                                                                                 │
│                 property_name='contract_type',                                                                  │
│                 operator=<ComparisonOperator.EQUALS: '='>,                                                      │
│                 value='service agreement'                                                                       │
│             )                                                                                                   │
│         ],                                                                                                      │
│         [                                                                                                       │
│             TextPropertyFilter(                                                                                 │
│                 property_name='contract_type',                                                                  │
│                 operator=<ComparisonOperator.EQUALS: '='>,                                                      │
│                 value='service agreement'                                                                       │
│             )                                                                                                   │
│         ],                                                                                                      │
│         [                                                                                                       │
│             TextPropertyFilter(                                                                                 │
│                 property_name='contract_type',                                                                  │
│                 operator=<ComparisonOperator.EQUALS: '='>,                                                      │
│                 value='service agreement'                                                                       │
│             )                                                                                                   │
│         ],                                                                                                      │
│         [                                                                                                       │
│             TextPropertyFilter(                                                                                 │
│                 property_name='contract_type',         

╭─────────────────────────────────────────── 🔭 Searches Executed 3/3 ────────────────────────────────────────────╮
│                                                                                                                 │
│ QueryResultWithCollection(                                                                                      │
│     queries=['common essential clauses in service agreements'],                                                 │
│     filters=[                                                                                                   │
│         [                                                                                                       │
│             TextPropertyFilter(                                                                                 │
│                 property_name='contract_type',                                                                  │
│                 operator=<ComparisonOperator.LIKE: 'LIKE'>,                                                     │
│                 value='*service agreement*'                                                                     │
│             )                                                                                                   │
│         ]                                                                                                       │
│     ],                                                                                                          │
│     filter_operators='AND',                                                                                     │
│     collection='Financialcontracts'                                                                             │
│ )                                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ 📊 No Aggregations Run                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── 📚 Sources ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  - object_id='668957da-cbd0-48cd-b096-fd0d08dea061' collection='Financialcontracts'                             │
│  - object_id='a185bbde-775a-4c55-9973-1e63e3e4d41b' collection='Financialcontracts'                             │
│  - object_id='dd4f37ec-6f30-45ed-b292-64538a7b2b2f' collection='Financialcontracts'                             │
│  - object_id='285d27af-c55c-4952-ac25-dfa0085999f6' collection='Financialcontracts'                             │
│  - object_id='bf49183c-6d0f-41c7-aa0a-8d2b8cfe8bb7' collection='Financialcontracts'                             │
│  - object_id='4212093e-1720-4405-96cf-635d41d66696' collection='Financialcontracts'                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   📊 Usage Statistics    
┌────────────────┬───────┐
│ LLM Requests:  │ 6     │
│ Input Tokens:  │ 17915 │
│ Output Tokens: │ 561   │
│ Total Tokens:  │ 18476 │
└────────────────┴───────┘

Total Time Taken: 10.06s

<a id='MultiStepWorkflows'></a>  
### 6.4 Multi-Step Agent Workflows  
[Back to table of contents](#TOC)

In [33]:
# Perform a comprehensive contract analysis
response = qa.run(
    "Find employment contracts in the database and analyze the main risks in these contracts, ranking them by severity. Provide specific examples from the contracts."
)

# Print the response
response.display()

╭─────────────────────────────────────────────── 🔍 Original Query ───────────────────────────────────────────────╮
│                                                                                                                 │
│ Find employment contracts in the database and analyze the main risks in these contracts, ranking them by        │
│ severity. Provide specific examples from the contracts.                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📝 Final Answer ────────────────────────────────────────────────╮
│                                                                                                                 │
│ A review of employment contracts in the database reveals several recurring risks. These can be ranked by        │
│ severity as follows:                                                                                            │
│                                                                                                                 │
│ 1. **Termination Provisions (High Severity)**                                                                   │
│    - Most contracts allow either party to terminate the contract with minimal notice, typically ranging from    │
│ two weeks to thirty days, and some permit termination "at any time, with or without cause." This can leave both │
│ employers and employees vulnerable to sudden changes, job/income loss, or resource gaps.                        │
│    - Example: “Either party may terminate this Contract at any time, with or without cause, by providing at     │
│ least two (2) weeks written notice…” (Edward Elric for Mark Robson).                                            │
│                                                                                                                 │
│ 2. **Ambiguous Bonus or Compensation Terms (Medium-High Severity)**                                             │
│    - Bonus payments are often described in vague terms, such as "may be eligible" or "may vary based on company │
│ performance," making it unclear whether employees can reliably expect such compensation.                        │
│    - Example: “Employee may also be eligible for bonuses based on performance, which can amount to an           │
│ additional $54,110 annually.” (John Smith contract) and “may be eligible for performance bonuses which may vary │
│ based on company performance…” (Edward Elric for Mark Robson).                                                  │
│                                                                                                                 │
│ 3. **Unclear Duties or Role Descriptions (Medium Severity)**                                                    │
│    - Several contracts use generic language for duties, such as "as assigned by the Employer," giving employers │
│ broad discretion to change roles or expectations without the employee's approval, which can lead to disputes.   │
│    - Example: “Employee agrees to perform the duties and responsibilities as assigned by the Employer…”         │
│ (multiple contracts).                                                                                           │
│                                                                                                                 │
│ 4. **Confidentiality Clauses and Post-Employment Restrictions (Medium Severity)**                               │
│    - While maintaining confidentiality is standard, some clauses obligate employees to continue these duties    │
│ indefinitely after termination, potentially affecting future job opportunities without clear limits.            │
│    - Example: “The Employee agrees to maintain the confidentiality of all proprietary information during and    │
│ after the term of employment.” (John Smith contract).                                                           │
│                                                                                                                 │
│ 5. **Benefits Subject to Change (Low-Medium Severity)**                                                         │
│    - Eligibility for benefits is typically described as "subject to the terms and conditions of those plans,"   │
│ which may change during employment, creating uncertainty.                                                       │
│    - Example: “Eligible to participate in the Employer'

╭─────────────────────────────────────────── 🔭 Searches Executed 1/2 ────────────────────────────────────────────╮
│                                                                                                                 │
│ QueryResultWithCollection(                                                                                      │
│     queries=['employment contract'],                                                                            │
│     filters=[                                                                                                   │
│         [                                                                                                       │
│             TextPropertyFilter(                                                                                 │
│                 property_name='contract_type',                                                                  │
│                 operator=<ComparisonOperator.LIKE: 'LIKE'>,                                                     │
│                 value='*employment*'                                                                            │
│             )                                                                                                   │
│         ]                                                                                                       │
│     ],                                                                                                          │
│     filter_operators='AND',                                                                                     │
│     collection='Financialcontracts'                                                                             │
│ )                                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🔭 Searches Executed 2/2 ────────────────────────────────────────────╮
│                                                                                                                 │
│ QueryResultWithCollection(                                                                                      │
│     queries=['employment contract main risks'],                                                                 │
│     filters=[                                                                                                   │
│         [                                                                                                       │
│             TextPropertyFilter(                                                                                 │
│                 property_name='contract_type',                                                                  │
│                 operator=<ComparisonOperator.LIKE: 'LIKE'>,                                                     │
│                 value='*employment*'                                                                            │
│             )                                                                                                   │
│         ]                                                                                                       │
│     ],                                                                                                          │
│     filter_operators='AND',                                                                                     │
│     collection='Financialcontracts'                                                                             │
│ )                                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 📊 Aggregations Run 1/2 ────────────────────────────────────────────╮
│                                                                                                                 │
│ AggregationResultWithCollection(                                                                                │
│     search_query=None,                                                                                          │
│     groupby_property=None,                                                                                      │
│     aggregations=[                                                                                              │
│         TextPropertyAggregation(                                                                                │
│             property_name='contract_type',                                                                      │
│             metrics=<TextMetrics.COUNT: 'COUNT'>,                                                               │
│             top_occurrences_limit=None                                                                          │
│         )                                                                                                       │
│     ],                                                                                                          │
│     filters=[                                                                                                   │
│         TextPropertyFilter(                                                                                     │
│             property_name='contract_type',                                                                      │
│             operator=<ComparisonOperator.EQUALS: '='>,                                                          │
│             value='employment contract'                                                                         │
│         )                                                                                                       │
│     ],                                                                                                          │
│     collection='Financialcontracts'                                                                             │
│ )                                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 📊 Aggregations Run 2/2 ────────────────────────────────────────────╮
│                                                                                                                 │
│ AggregationResultWithCollection(                                                                                │
│     search_query=None,                                                                                          │
│     groupby_property='contract_type',                                                                           │
│     aggregations=[                                                                                              │
│         TextPropertyAggregation(                                                                                │
│             property_name='contract_text',                                                                      │
│             metrics=<TextMetrics.TOP_OCCURRENCES: 'TOP_OCCURRENCES'>,                                           │
│             top_occurrences_limit=5                                                                             │
│         )                                                                                                       │
│     ],                                                                                                          │
│     filters=[                                                                                                   │
│         TextPropertyFilter(                                                                                     │
│             property_name='contract_type',                                                                      │
│             operator=<ComparisonOperator.LIKE: 'LIKE'>,                                                         │
│             value='*employment*'                                                                                │
│         )                                                                                                       │
│     ],                                                                                                          │
│     collection='Financialcontracts'                                                                             │
│ )                                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── 📚 Sources ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  - object_id='3d078c02-09f3-4aec-9c89-3b0b3fada95b' collection='Financialcontracts'                             │
│  - object_id='6731310b-a9a1-4e13-a954-cb7bf246b970' collection='Financialcontracts'                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   📊 Usage Statistics    
┌────────────────┬───────┐
│ LLM Requests:  │ 7     │
│ Input Tokens:  │ 23020 │
│ Output Tokens: │ 867   │
│ Total Tokens:  │ 23887 │
└────────────────┴───────┘

Total Time Taken: 16.04s

In [39]:
# close the Weaviate client to free up resources
client.close()